# 53 · Detect unusual counts, against EARS, CUSUM and EWMA in numpy

**Question.** Fieldwork's Detect unusual counts holds each period of a series of counts to a threshold from the series'
own past, by EARS C1, C2 or C3, CUSUM or EWMA. Written again here from the published definitions, do the five methods
give the same expected counts, thresholds, statistics, limits and signals, period by period, on four series: 40 made-up
days with an excess, the syndromic example's gastrointestinal days, three seasonal years, and a sparse series?

**Here.** numpy does the arithmetic and scipy gives the normal quantile; nothing of Fieldwork's code is read. On the 40
days, R's `surveillance` 1.25.0 is read a third time from its recorded values.

In [ ]:
NEEDED = ["numpy", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

## Parity

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_53_detect_unusual_counts import run

report = run(Path.cwd() / "fixtures" / "detect-unusual-counts-001.json")
print("agrees:", report["agrees"])
for r in report["runs"]:
    said = f"{r['signals']} signal" if "signals" in r else f"refused: {r['refused']['here']}"
    print(f"  {'ok' if r['agrees'] else 'NO'}  {r['run']}: {said}", *r["differs"][:3])

## Against R, on the 40 days

C1, C2 and CUSUM are judged. C3 is reported only: the paper counts the current day in C3, and R's bound leaves it out,
so R cannot signal on the first day of a rise.

In [ ]:
for method, v in report["againstR"].items():
    print(f"{method:8} largest threshold difference {v['largestThresholdDifference']}")
    print(f"         signals here {v['signals']['here']}")
    print(f"         signals in R {v['signals']['R']}")

## What the seasonal series shows

The peak is each January. CUSUM, held to the first year's mean, signals for weeks on end in each later winter; EWMA,
whose limit the first year's spread sets, and the season widens, signals in no week. The EARS methods signal on every
autumn's rise, each year: an expected rise looks unusual against the weeks just before it. None of these is a seasonal
baseline, which would hold each week to the same weeks of past years.

In [ ]:
for r in report["runs"]:
    if r["run"].startswith("seasonal"):
        print(f"{r['run']}: {r.get('signals')} of {r.get('periods')} weeks signal")

## What this cannot establish

**That a signal is an outbreak**, or **that any α, h or L suits a surveillance system**: both sides compute the same
statistic from the same declared baseline. Nor a seasonal baseline (Farrington), which Fieldwork does not have.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("numpy   :", e["numpy"], "· scipy", e["scipy"])
print("runtime :", e["runtime"], "· Pyodide" if e["pyodide"] else "")